In [ ]:
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from transformers import pipeline

model_name = "enstazao/Qalb-1.0-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto"  # <--- Required for quantization
)


urdu_system_prompt = "آپ ایک مددگار اور بے ضرر مصنوعی ذہانت کے اسسٹنٹ ہیں۔ آپ اردو میں سوالات کے درست جوابات دیتے ہیں۔"
terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]


def generate_answer(problem):

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

    {urdu_system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

    {problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
    """

    input_ids = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **input_ids,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.1,
        do_sample=True,
        eos_token_id=terminators,
    )

    response = tokenizer.decode(
        outputs[0][input_ids["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response

In [ ]:
dataset = load_dataset("large-traversaal/mgsm_urdu_final", split="test")
results = []

for row in tqdm(dataset):
    response = generate_answer(row["urdu_question"])  # Your model response

    results.append(
        {
            "question": row["urdu_question"],
            "answer": row["answer_number"],
            "response": response,
        }
    )

df = pd.DataFrame(results)
df.to_csv("qalb-Instruct-response.csv", index=False)

In [ ]:
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from transformers import pipeline

df = pd.read_csv("qalb-Instruct-response.csv")

validation_results = []

model_id = "openai/gpt-oss-20b"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto",
)

for _, row in tqdm(df.iterrows()):

    prompt = f"""You are an answer validator for Urdu math reasoning problems.
    
    Input:
    
    * Question: {row['question']}
    * Solution: {row['response']}
    * Expected Answer: {row['answer']}
    
    Task:
    
    1. Read the Urdu math problem and the provided solution.
    2. Determine the final answer produced by the solution.
    3. Compare it with the Expected Answer.
    4. Treat numerically equivalent values as equal:
    
       * 2 = 2.0 = 2.00
       * 5 = 5.000
       * Ignore insignificant trailing zeros in decimals.
    5. If both answers represent the same numeric value, return:
       True
    6. Otherwise return:
       False
    
    Rules:
    
    * Return ONLY one word: True or False.
    * Do not provide explanations, reasoning, calculations, or extra text.
    * Comparison should be based on the final answer only.
    * For numeric answers, compare by value, not string format.
    * If the solution's final answer cannot be determined with confidence, return False.
     """

    messages = [{"role": "user", "content": prompt}]
    outputs = pipe(messages, max_new_tokens=1000)

    result = (
        True
        if "True.assistantfinalTrue" in outputs[0]["generated_text"][-1]["content"]
        else False
    )
    print(outputs[0]["generated_text"][-1]["content"])
    print(result)
    print("\n\n")
    validation_results.append(result)

df["is_correct"] = validation_results

df.to_csv("qalb-Instruct_correct.csv", index=False)

print(df.head())